# v14a — Unified Injection Corpus Builder

**No model required.** Pure pandas. Produces `datasets/unified/injection_corpus.csv`,
the single input for v14b (extraction) and v14c (analysis).

## Why this exists

Five corpora, four attack families, two delivery modes. They cannot be concatenated
naively — three things would break.

**1. Count is not diversity.** BIPIA has **250 unique payloads** that expand
combinatorially to 67,500 examples (900 table contexts × 75 payloads). InjecAgent has
**62 payloads** → 2,108 cases. Grouping by text similarity does **not** catch this: the
carrier document dominates the n-grams, so the same payload in two different emails looks
unrelated. Every row therefore carries an explicit **`payload_id`**, and that is the CV
group key — not text similarity.

**2. Indirect injections need matched clean carriers.** For every BIPIA/InjecAgent
injected example this builder emits the *same carrier with no payload* as a negative,
sharing its `carrier_id`. The detector then has to find the payload, not recognise the
document. This is the cleanest hard-negative design available and it attacks the confound
that has blocked every version since 02.

**3. Length was a shortcut.** On v13's matched pair, character count alone gives
AUC 0.793. Merging corpora breaks that correlation — short injections (DeepSet 124) sit
alongside long ones (SPML 410, BIPIA 500–1,600), and short benign (DeepSet 42) alongside
long (WJB-adv 726). Every row carries `n_chars` so v14c can also build length-matched
subsets and prove it.

## Schema

| column | meaning |
|---|---|
| `text` | the exact string fed to the model |
| `label` | 1 injection · 0 benign |
| `corpus` | DeepSet · SPML · BIPIA · InjecAgent · WJB-Adv-Benign · … |
| `family` | direct · indirect · benign |
| `payload_id` | **CV group key** — never split across folds |
| `carrier_id` | shared by an injected example and its matched clean carrier |
| `lang`, `n_chars`, `meta` | diagnostics |

## Grouping rules (stated so they can be defended)
- **BIPIA / InjecAgent** → `payload_id` = the actual payload index. Exact and correct.
- **DeepSet** → near-duplicate cluster (4-gram Jaccard ≥ 0.40). n=263, O(n²) is fine.
- **SPML** → grouped by **System Prompt** (840 unique). Attacks written against the same
  system prompt are related by construction; this is cheaper and more faithful than text
  similarity over 12,541 rows.
- **Benign** → singleton groups, except BIPIA/InjecAgent carriers which inherit `carrier_id`.

## Modelling decision to state in the thesis
BIPIA and InjecAgent deliver the payload through retrieved content or tool output, not the
user turn. The monitor formats `system + user`, so carrier+payload is placed in the user
turn — as if the person pasted a document or the tool result was inlined. Standard
simplification, but say it in the methods rather than letting an examiner find it.

## 1 · Config

In [17]:
import os, re, json, hashlib, itertools
import numpy as np
import pandas as pd

SEED = 42
rng  = np.random.default_rng(SEED)

DATA = "datasets"
OUT  = os.path.join(DATA, "unified")
os.makedirs(OUT, exist_ok=True)

# ---- caps: lower these if extraction time is tight -------------------------
CAP = dict(
    SPML_INJ            = 2500,   # of 12,541 unique
    SPML_BEN            = 1200,   # of 3,372
    BIPIA_PER_PAYLOAD   = 3,      # carriers per payload -> 250 x 3 injected + 3 matched clean
    INJAG_PER_PAYLOAD   = 3,      # user cases per payload -> 62 x 3
    WJB_ADV_BEN         = 2500,   # of 78,731  <- the hard negative class
    WJB_VAN_BEN         = 800,
    XSTEST_SAFE         = 250,
    JBB_BEN             = 100,
    DOLLY_BASELINE      = 2000,   # PCA baseline pool (v13 used 100 -> PCA capped at 99)
)
DUP_JACCARD = 0.40
# Carrier budget: v14b truncates at MAX_LEN tokens. A BIPIA payload sitting at the END
# of a long carrier gets cut off -- an example labelled "injection" that contains none
# (129 rows, 3.1% of positives, at MAX_LEN=512). Truncating the CARRIER keeps the payload.
MAX_CHARS = 3000   # ~1024 tokens; keep in step with MAX_LEN in v14b
print("v14a corpus builder | seed", SEED)
for k, v in CAP.items(): print(f"  {k:<20}{v}")

v14a corpus builder | seed 42
  SPML_INJ            2500
  SPML_BEN            1200
  BIPIA_PER_PAYLOAD   3
  INJAG_PER_PAYLOAD   3
  WJB_ADV_BEN         2500
  WJB_VAN_BEN         800
  XSTEST_SAFE         250
  JBB_BEN             100
  DOLLY_BASELINE      2000


## 2 · Helpers

In [18]:
def clean_text(t):
    # identical to Preprocessor.clean_text in main.ipynb cell [6]
    t = re.sub(r"\s+", " ", str(t).strip())
    return re.sub(r"[^\x00-\x7F]+", "", t)

def dedup_keep_order(xs):
    seen, out = set(), []
    for x in xs:
        k = x.strip().lower()
        if k and k not in seen:
            seen.add(k); out.append(x)
    return out

def jaccard_clusters(texts, th=DUP_JACCARD, n=4):
    # union-find over 4-gram Jaccard. O(n^2) -- only use on small sets.
    G = [{s.lower()[i:i+n] for i in range(max(1, len(s)-n+1))} for s in texts]
    N = len(texts); par = list(range(N))
    def find(x):
        while par[x] != x:
            par[x] = par[par[x]]; x = par[x]
        return x
    for i in range(N):
        for j in range(i+1, N):
            u = len(G[i] | G[j])
            if u and len(G[i] & G[j]) / u >= th:
                a, b = find(i), find(j)
                if a != b: par[a] = b
    roots = {}
    return [roots.setdefault(find(i), len(roots)) for i in range(N)]

DE_RE = re.compile(r"\b(der|die|das|und|ist|nicht|du|sie|wie|was|warum|bitte|ich|alle|"
                   r"vorherigen|anweisungen|vergiss|ignoriere|jetzt|eine|einen|mir|mich|"
                   r"wir|sind|haben|werden|schreibe|fr|ber|mit|auf|von|zu|dich|deine|dass)\b", re.I)
lang_of = lambda t: "de" if len(DE_RE.findall(t)) >= 2 else "en"

ROWS = []
def add(text, label, corpus, family, payload_id, carrier_id=None, meta=""):
    t = clean_text(text)
    if not t: return
    ROWS.append(dict(text=t, label=int(label), corpus=corpus, family=family,
                     payload_id=f"{corpus}:{payload_id}",
                     carrier_id=(f"{corpus}:{carrier_id}" if carrier_id is not None else ""),
                     lang=lang_of(t), n_chars=len(t), meta=meta))

def take(seq, k):
    seq = list(seq)
    if k is None or k >= len(seq): return seq
    idx = rng.choice(len(seq), size=k, replace=False)
    return [seq[i] for i in sorted(idx)]

## 3 · Direct injections — DeepSet + SPML

In [19]:
# ---- DeepSet (263 injections / 399 benign, EN+DE) --------------------------
d = pd.read_csv(f"{DATA}/deepset_prompt_injections.csv")
ds_inj = dedup_keep_order([clean_text(x) for x in d[d.label == 1].text.dropna()])
ds_ben = dedup_keep_order([clean_text(x) for x in d[d.label == 0].text.dropna()])[:300]
cl = jaccard_clusters(ds_inj)
for t, c in zip(ds_inj, cl): add(t, 1, "DeepSet", "direct", f"cl{c}")
for i, t in enumerate(ds_ben): add(t, 0, "DeepSet-Benign", "benign", f"b{i}")
print(f"DeepSet      {len(ds_inj):>5} injections ({len(set(cl))} payload groups) | {len(ds_ben)} benign")

# ---- SPML (12,541 inj / 3,372 ben; grouped by System Prompt) ---------------
s = pd.read_csv(f"{DATA}/spml_prompt_injection.csv")
s = s.dropna(subset=["User Prompt"])
s["_t"] = s["User Prompt"].map(clean_text)
s = s[s._t.str.len() > 0].drop_duplicates("_t")
s["_g"] = s["System Prompt"].fillna("NA").map(lambda x: hashlib.md5(str(x).encode()).hexdigest()[:10])

si = s[s["Prompt injection"] == 1]
sb = s[s["Prompt injection"] == 0]
# stratify the injection sample by Degree so attack intensity stays spread
si_s = (si.groupby("Degree", group_keys=False)
          .apply(lambda g: g.sample(min(len(g), max(1, CAP["SPML_INJ"] // max(1, si.Degree.nunique()))),
                                    random_state=SEED))
        if "Degree" in si.columns else si.sample(min(len(si), CAP["SPML_INJ"]), random_state=SEED))
si_s = si_s.sample(min(len(si_s), CAP["SPML_INJ"]), random_state=SEED)
sb_s = sb.sample(min(len(sb), CAP["SPML_BEN"]), random_state=SEED)

for _, r in si_s.iterrows():
    add(r._t, 1, "SPML", "direct", r._g, meta=f"degree={r.get('Degree','')}")
for _, r in sb_s.iterrows():
    add(r._t, 0, "SPML-Benign", "benign", r._g)
print(f"SPML         {len(si_s):>5} injections ({si_s._g.nunique()} payload groups) | {len(sb_s)} benign")

DeepSet        263 injections (172 payload groups) | 300 benign


/var/folders/s9/2gcqlc9d6c50xyx2k7j4zsbw0000gn/T/ipykernel_87333/1639153265.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), max(1, CAP["SPML_INJ"] // max(1, si.Degree.nunique()))),


SPML          2499 injections (814 payload groups) | 1200 benign


## 4 · Indirect injections — BIPIA

Carrier + payload assembled here. **Every injected example also emits its clean carrier as a matched negative** sharing `carrier_id`, so the detector must find the payload rather than recognise the document.

In [20]:
def load_jsonl(p):
    return [json.loads(l) for l in open(p) if l.strip()]

BIPIA_TASKS = {  # task -> (context file, attack file, how to render the carrier)
    "email": ("BIPIA/email/{s}.jsonl", "BIPIA/text_attack_{s}.json"),
    "table": ("BIPIA/table/{s}.jsonl", "BIPIA/text_attack_{s}.json"),
    "code":  ("BIPIA/code/{s}.jsonl",  "BIPIA/code_attack_{s}.json"),
}

def render_carrier(task, row):
    if task == "code":
        return " ".join(str(row.get(k, "")) for k in ("error", "code", "context") if row.get(k))
    return f"{row.get('context','')}\n\n{row.get('question','')}"

n_inj = n_ben = 0
seen_carrier = set()
for task, (cpat, apat) in BIPIA_TASKS.items():
    for split in ("train", "test"):
        cp, ap = f"{DATA}/{cpat.format(s=split)}", f"{DATA}/{apat.format(s=split)}"
        if not (os.path.exists(cp) and os.path.exists(ap)):
            print(f"  [skip] {task}/{split} missing"); continue
        ctxs = load_jsonl(cp)
        atk  = json.load(open(ap))                       # {attack_type: [payload, ...]}
        payloads = [(f"{task}-{split}-{t}-{i}", t, p)
                    for t, lst in atk.items() for i, p in enumerate(lst)]
        for pid, atype, payload in payloads:
            for row in take(ctxs, CAP["BIPIA_PER_PAYLOAD"]):
                carrier = render_carrier(task, row)
                if not carrier.strip(): continue
                budget = MAX_CHARS - len(payload) - 4
                if len(carrier) > budget:          # keep the payload, cut the carrier
                    carrier = carrier[:max(0, budget)]
                cid = hashlib.md5(carrier.encode()).hexdigest()[:10]
                pos = rng.integers(0, 2)                  # payload at start or end
                text = f"{payload}\n\n{carrier}" if pos == 0 else f"{carrier}\n\n{payload}"
                add(text, 1, "BIPIA", "indirect", pid, cid,
                    meta=f"task={task};type={atype};pos={'start' if pos==0 else 'end'}")
                n_inj += 1
                if cid not in seen_carrier:               # matched clean carrier, once each
                    add(carrier, 0, "BIPIA-Clean", "benign", f"clean-{cid}", cid, meta=f"task={task}")
                    seen_carrier.add(cid); n_ben += 1
print(f"BIPIA        {n_inj:>5} injections | {n_ben} matched clean carriers")

BIPIA         1200 injections | 536 matched clean carriers


## 5 · Indirect injections — InjecAgent

The attacker instruction is substituted into each user case's `Tool Response Template`. The same template with a neutral string is the matched negative.

In [21]:
uc = load_jsonl(f"{DATA}/InjectAgent/user_cases.jsonl")
atk = (load_jsonl(f"{DATA}/InjectAgent/attacker_cases_dh.jsonl")
       + load_jsonl(f"{DATA}/InjectAgent/attacker_cases_ds.jsonl"))
NEUTRAL = "Great product, arrived on time and works exactly as described."

n_inj = n_ben = 0
seen_case = set()
for ai, a in enumerate(atk):
    instr = a.get("Attacker Instruction", "")
    if not instr: continue
    for u in take(uc, CAP["INJAG_PER_PAYLOAD"]):
        tmpl = str(u.get("Tool Response Template", ""))
        if "<Attacker Instruction>" not in tmpl: continue
        cid  = hashlib.md5(tmpl.encode()).hexdigest()[:10]
        text = (f"{u.get('User Instruction','')}\n\n"
                f"Tool result ({u.get('User Tool','tool')}): "
                f"{tmpl.replace('<Attacker Instruction>', instr)}")
        add(text, 1, "InjecAgent", "indirect", f"atk{ai}", cid,
            meta=f"type={a.get('Attack Type','')};tool={u.get('User Tool','')}")
        n_inj += 1
        if cid not in seen_case:
            add(f"{u.get('User Instruction','')}\n\n"
                f"Tool result ({u.get('User Tool','tool')}): "
                f"{tmpl.replace('<Attacker Instruction>', NEUTRAL)}",
                0, "InjecAgent-Clean", "benign", f"clean-{cid}", cid,
                meta=f"tool={u.get('User Tool','')}")
            seen_case.add(cid); n_ben += 1
print(f"InjecAgent   {n_inj:>5} injections ({len(set(a.get('Attacker Instruction') for a in atk))} unique payloads)"
      f" | {n_ben} matched clean")

InjecAgent     186 injections (62 unique payloads) | 17 matched clean


## 6 · Benign corpora + PCA baseline pool

In [22]:
w = pd.read_csv(f"{DATA}/wildjailbreak_train.csv", sep=None, engine="python", on_bad_lines="skip")
for dt, cap, name in [("adversarial_benign", CAP["WJB_ADV_BEN"], "WJB-Adv-Benign"),
                      ("vanilla_benign",     CAP["WJB_VAN_BEN"], "WJB-Vanilla-Benign")]:
    col = "adversarial" if dt.startswith("adversarial") else "vanilla"
    pool = dedup_keep_order([clean_text(x) for x in w[w.data_type == dt][col].dropna()])
    for i, t in enumerate(take(pool, cap)): add(t, 0, name, "benign", f"b{i}")
    print(f"{name:<20}{min(cap, len(pool)):>5} of {len(pool):,} available")

x = pd.read_csv(f"{DATA}/xstest_prompts.csv")
tcol = "prompt" if "prompt" in x.columns else x.columns[-1]
safe = x[~x.get("type", pd.Series([""]*len(x))).astype(str).str.contains("contrast", case=False, na=False)]
for i, t in enumerate(take([clean_text(v) for v in safe[tcol].dropna()], CAP["XSTEST_SAFE"])):
    add(t, 0, "XSTest-Safe", "benign", f"b{i}")

j = pd.read_csv(f"{DATA}/jailbreakbench/benign-behaviors.csv")
jc = "Goal" if "Goal" in j.columns else j.columns[0]
for i, t in enumerate(take([clean_text(v) for v in j[jc].dropna()], CAP["JBB_BEN"])):
    add(t, 0, "JBB-Benign", "benign", f"b{i}")

# baseline pool: NOT a negative -- it is what PCA is fitted on.
dol = pd.read_csv(f"{DATA}/dolly-15k-instruction-alpaca-format.csv")
dc = "instruction" if "instruction" in dol.columns else dol.columns[0]
base = take(dedup_keep_order([clean_text(v) for v in dol[dc].dropna()]), CAP["DOLLY_BASELINE"])
for i, t in enumerate(base): add(t, 0, "Baseline-Dolly", "baseline", f"b{i}")
print(f"{'XSTest/JBB/Baseline':<20}added (baseline pool = {len(base)}; v13 used 100 -> PCA capped at 99)")

WJB-Adv-Benign       2500 of 78,705 available
WJB-Vanilla-Benign    800 of 49,952 available
XSTest/JBB/Baseline added (baseline pool = 2000; v13 used 100 -> PCA capped at 99)


## 7 · Assemble, cross-corpus dedup, write

In [23]:
df = pd.DataFrame(ROWS)
before = len(df)
df["_k"] = df.text.str.strip().str.lower()
df = df.sort_values(["label", "corpus"]).drop_duplicates("_k", keep="first").drop(columns="_k")
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print(f"cross-corpus dedup: {before:,} -> {len(df):,} rows ({before-len(df)} duplicates removed)")

path = f"{OUT}/injection_corpus.csv"
df.to_csv(path, index=False)
manifest = dict(seed=SEED, caps=CAP, dup_jaccard=DUP_JACCARD, rows=int(len(df)),
                by_corpus=df.corpus.value_counts().to_dict(),
                by_family=df.family.value_counts().to_dict(),
                payload_groups=int(df.payload_id.nunique()))
json.dump(manifest, open(f"{OUT}/manifest.json", "w"), indent=2)
print("wrote", path, "and manifest.json")

cross-corpus dedup: 11,851 -> 11,838 rows (13 duplicates removed)
wrote datasets/unified/injection_corpus.csv and manifest.json


## 8 · Sanity report — read this before extracting

In [24]:
ev = df[df.family != "baseline"]
print(f"{'corpus':<22}{'label':>6}{'rows':>7}{'payload groups':>16}{'med chars':>11}")
print("-"*64)
for (c, l), g in ev.groupby(["corpus", "label"]):
    print(f"{c:<22}{l:>6}{len(g):>7}{g.payload_id.nunique():>16}{int(g.n_chars.median()):>11}")
print("-"*64)
print(f"{'TOTAL injections':<22}{1:>6}{(ev.label==1).sum():>7}"
      f"{ev[ev.label==1].payload_id.nunique():>16}")
print(f"{'TOTAL benign':<22}{0:>6}{(ev.label==0).sum():>7}"
      f"{ev[ev.label==0].payload_id.nunique():>16}")
print(f"{'baseline pool':<22}{'-':>6}{(df.family=='baseline').sum():>7}")

print("\nby family:"); print(ev.groupby(["family","label"]).size().to_string())

# the shortcut that has to be dead
def auc_len(pos, neg):
    v = np.r_[pos.values, neg.values]; n1, n0 = len(pos), len(neg)
    r = pd.Series(v).rank().values
    a = (r[:n1].sum() - n1*(n1+1)/2) / (n1*n0); return max(a, 1-a)
pos, neg = ev[ev.label==1].n_chars, ev[ev.label==0].n_chars
print(f"\nLENGTH-ONLY AUC (injection vs benign): {auc_len(pos, neg):.3f}")
print("  v13's matched pair was 0.793 -- lower is better; near 0.5 means length is dead.")

g = ev.payload_id.value_counts()
print(f"\npayload group sizes: {len(g)} groups, max {g.max()}, "
      f"{(g>1).sum()} groups with >1 row ({g[g>1].sum()} rows)")
print("  v14c MUST use payload_id as the CV group key. Text clustering will not catch these.")

n_extract = len(df)
print(f"\nEXTRACTION COST: {n_extract:,} prompts x 8 layers.")
print("  Time a batch of 50 in v14b first, then multiply -- lower CAP values if it is too slow.")

corpus                 label   rows  payload groups  med chars
----------------------------------------------------------------
BIPIA                      1   1200             400        839
BIPIA-Clean                0    525             525        938
DeepSet                    1    263             172        124
DeepSet-Benign             0    300             300         42
InjecAgent                 1    186              62        377
InjecAgent-Clean           0     17              17        317
JBB-Benign                 0    100             100         73
SPML                       1   2499             814        501
SPML-Benign                0   1198             608         76
WJB-Adv-Benign             0   2500            2500        726
WJB-Vanilla-Benign         0    800             800         76
XSTest-Safe                0    250             250         43
----------------------------------------------------------------
TOTAL injections           1   4148            1448

## Next

1. Check the report above — especially **length-only AUC** and **payload group sizes**.
2. If the row count is too large for one extraction session, lower `CAP` and re-run. This
   notebook is deterministic under `SEED`, so the same caps always give the same corpus.
3. Then **v14b** (extraction, needs the model, checkpointed) and **v14c** (the four
   experiments: payload-grouped CV, leave-one-corpus-out, length-matched controls,
   separability ceiling).

`or-bench-hard-1k` is not wired in — you have not downloaded it. It would add a second
independent hard-negative family; add it to §6 if you fetch it.